In [ ]:
!pip install monai SimpleITK lifelines scikit-survival -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from google.colab import drive

drive.mount('/content/drive')

BASE   = Path('/content/drive/MyDrive/multimodal_prognosis/')
TENSOR = BASE / 'tensors'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
class CTSurvivalDataset(Dataset):
    def __init__(self, labels_csv, tensor_dir):
        self.df         = pd.read_csv(labels_csv)
        self.tensor_dir = Path(tensor_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        ct_tensor  = torch.load(self.tensor_dir / f"{row['patient_id']}.pt").float()

        # Tabular features: age, stage, gender
        tabular = torch.tensor([
            row['age'] / 100.0,        # normalise age
            row['stage'] / 4.0,        # normalise stage
            float(row['gender']),
        ], dtype=torch.float32)

        time  = torch.tensor(row['survival_days'], dtype=torch.float32)
        event = torch.tensor(row['event'],         dtype=torch.float32)

        return ct_tensor, tabular, time, event

dataset = CTSurvivalDataset(BASE / 'NSCLC-Radiomics-Lung1', TENSOR)
print(f"Dataset size: {len(dataset)}")

ct, tab, t, e = dataset[0]
print(f"CT shape    : {ct.shape}")
print(f"Tabular     : {tab}")
print(f"Time/Event  : {t.item():.0f} days, event={e.item():.0f}")

In [ ]:
from torch.utils.data import random_split

torch.manual_seed(42)
n       = len(dataset)   # 200
n_train = 140            # 70%
n_val   = 30             # 15%
n_test  = 30             # 15%

train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
class ResBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class ImageEncoder(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            # 1 -> 16 channels, downsample to 32^3
            nn.Conv3d(1, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(16), nn.ReLU(inplace=True),
            ResBlock3D(16),

            # 16 -> 32 channels, downsample to 16^3
            nn.Conv3d(16, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            ResBlock3D(32),

            # 32 -> 64 channels, downsample to 8^3
            nn.Conv3d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            ResBlock3D(64),

            nn.AdaptiveAvgPool3d(1),   # -> (B, 64, 1, 1, 1)
            nn.Flatten(),              # -> (B, 64)
        )
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.encoder(x))



enc = ImageEncoder(out_dim=64).to(DEVICE)
dummy = torch.randn(2, 1, 64, 64, 64).to(DEVICE)
out   = enc(dummy)
print(f"ImageEncoder output shape: {out.shape}")  # expect (2, 64)

In [ ]:
class TabularEncoder(nn.Module):
    def __init__(self, in_dim=3, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(64, out_dim),
        )

    def forward(self, x):
        return self.net(x)

tab_enc = TabularEncoder(in_dim=3, out_dim=64).to(DEVICE)
dummy_t = torch.randn(2, 3).to(DEVICE)
print(f"TabularEncoder output shape: {tab_enc(dummy_t).shape}")  # expect (2, 64)

In [ ]:
def cox_loss(risk_scores, times, events):
    """
    Negative partial log-likelihood for Cox proportional hazards.
    risk_scores : (B,) — higher = higher risk
    times       : (B,) — survival time
    events      : (B,) — 1 = event occurred, 0 = censored
    """
    # sort by descending survival time
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events      = events[order]

    # log-sum-exp of risk scores for the risk set at each event time
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    loss        = -torch.mean((risk_scores - log_cumsum) * events)
    return loss


def concordance_index(risk_scores, times, events):
    """Simple C-index: fraction of concordant pairs among comparable pairs."""
    risk_scores = risk_scores.cpu().numpy()
    times       = times.cpu().numpy()
    events      = events.cpu().numpy()

    concordant = 0
    comparable = 0
    for i in range(len(times)):
        for j in range(len(times)):
            if events[i] == 1 and times[i] < times[j]:
                comparable += 1
                if risk_scores[i] > risk_scores[j]:
                    concordant += 1
                elif risk_scores[i] == risk_scores[j]:
                    concordant += 0.5

    return concordant / comparable if comparable > 0 else 0.5

In [ ]:
class ImageOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder(out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, ct, _tab):
        feat = self.encoder(ct)
        return self.head(feat).squeeze(-1)  # (B,)

In [ ]:
class TabularOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TabularEncoder(in_dim=3, out_dim=64)
        self.head    = nn.Linear(64, 1)

    def forward(self, _ct, tab):
        feat = self.encoder(tab)
        return self.head(feat).squeeze(-1)  # (B,)

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=50, lr=1e-3, save_path=None):

    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=5)

    history = {'train_loss': [], 'val_cindex': []}
    best_ci  = 0.0

    for epoch in range(epochs):

        # train 
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for ct, tab, time, event in train_loader:
            ct, tab     = ct.to(DEVICE), tab.to(DEVICE)
            time, event = time.to(DEVICE), event.to(DEVICE)

            opt.zero_grad()
            risk = model(ct, tab)
            loss = cox_loss(risk, time, event)

            if torch.isnan(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / max(n_batches, 1)

        # validate 
        model.eval()
        all_risk, all_time, all_event = [], [], []
        with torch.no_grad():
            for ct, tab, time, event in val_loader:
                ct, tab = ct.to(DEVICE), tab.to(DEVICE)
                all_risk.append(model(ct, tab).cpu())
                all_time.append(time)
                all_event.append(event)

        risk_cat  = torch.cat(all_risk)
        time_cat  = torch.cat(all_time)
        event_cat = torch.cat(all_event)
        val_ci    = concordance_index(risk_cat, time_cat, event_cat)

        sched.step(val_ci)
        history['train_loss'].append(avg_loss)
        history['val_cindex'].append(val_ci)

        # save best checkpoint
        if val_ci > best_ci and save_path:
            best_ci = val_ci
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | "
                  f"Val C-index: {val_ci:.4f} | Best: {best_ci:.4f}")

    # load best weights before returning
    if save_path and Path(save_path).exists():
        model.load_state_dict(torch.load(save_path))

    return history, best_ci

In [ ]:
print("=" * 50)
print("Training image-only baseline")
print("=" * 50)

image_model = ImageOnlyModel().to(DEVICE)
history_img, best_img_ci = train_model(
    image_model, train_loader, val_loader,
    epochs=50, lr=1e-3,
    save_path=BASE / 'image_only_best.pt'
)

print(f"\nImage-only best val C-index: {best_img_ci:.4f}")

In [ ]:
print("=" * 50)
print("Training tabular-only baseline")
print("=" * 50)

tab_model = TabularOnlyModel().to(DEVICE)
history_tab, best_tab_ci = train_model(
    tab_model, train_loader, val_loader,
    epochs=50, lr=1e-3,
    save_path=BASE / 'tabular_only_best.pt'
)

print(f"\nTabular-only best val C-index: {best_tab_ci:.4f}")

In [ ]:
def evaluate_on_test(model, test_loader):
    model.eval()
    all_risk, all_time, all_event = [], [], []
    with torch.no_grad():
        for ct, tab, time, event in test_loader:
            ct, tab = ct.to(DEVICE), tab.to(DEVICE)
            all_risk.append(model(ct, tab).cpu())
            all_time.append(time)
            all_event.append(event)

    risk_cat  = torch.cat(all_risk)
    time_cat  = torch.cat(all_time)
    event_cat = torch.cat(all_event)
    return concordance_index(risk_cat, time_cat, event_cat)


test_ci_img = evaluate_on_test(image_model, test_loader)
test_ci_tab = evaluate_on_test(tab_model,   test_loader)

print("=" * 50)
print("TEST SET RESULTS — baselines")
print("=" * 50)
print(f"Image-only   test C-index : {test_ci_img:.4f}")
print(f"Tabular-only test C-index : {test_ci_tab:.4f}")
print(f"Random baseline           : 0.5000")
print()
print("These are the numbers fusion must beat in notebook 3.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history_img['train_loss']) + 1)

axes[0].plot(epochs, history_img['train_loss'], color='steelblue', label='Image-only')
axes[0].plot(epochs, history_tab['train_loss'], color='coral',     label='Tabular-only')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cox loss')
axes[0].set_title('Training loss')
axes[0].legend()

axes[1].plot(epochs, history_img['val_cindex'], color='steelblue', label='Image-only')
axes[1].plot(epochs, history_tab['val_cindex'], color='coral',     label='Tabular-only')
axes[1].axhline(0.5, color='gray', linestyle='--', label='Random (0.5)')
axes[1].axhline(test_ci_img, color='steelblue', linestyle=':', alpha=0.6,
                label=f'Test C-index img={test_ci_img:.3f}')
axes[1].axhline(test_ci_tab, color='coral',     linestyle=':', alpha=0.6,
                label=f'Test C-index tab={test_ci_tab:.3f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('C-index')
axes[1].set_title('Validation C-index')
axes[1].legend(fontsize=8)

plt.suptitle('Baseline models — 200 patients, 70/15/15 split', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'baselines_200.png', dpi=150, bbox_inches='tight')
plt.show()